In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, X_feat_train, X_feat_test = load_cache()
tf.keras.utils.set_random_seed(SEED)  # reproducible weights init / shuffling

Loading cached dataset from ptbxl_preprocessed_cache.npz...
Loaded successfully in seconds!
Signals   -> Train: (19230, 1000, 3) | Test: (2158, 1000, 3)
Features  -> Train: (19230, 36) | Test: (2158, 36)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [4]:
def build_lightweight_1d_cnn(input_shape=(1000, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape, name="ecg_input")
    
    # Block 1: Initial temporal reduction
    x = layers.Conv1D(16, kernel_size=7, strides=2, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # Block 2: Depthwise Separable Conv
    x = layers.SeparableConv1D(32, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # Block 3: Depthwise Separable Conv
    x = layers.SeparableConv1D(64, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    # Global Average Pooling replaces parameter-heavy Flatten layers
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="sigmoid", name="class_outputs")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="TinyML_1D_CNN")

cnn_1d = build_lightweight_1d_cnn(input_shape=(X_train.shape[1], X_train.shape[2]), num_classes=len(class_names))
cnn_1d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")]
)
cnn_1d.summary()

cb = [
    callbacks.EarlyStopping(monitor="val_macro_auc", mode="max", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]

print("\nTraining Lightweight 1D-CNN...")
history_1d = cnn_1d.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    verbose=1
)

Model: "TinyML_1D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 1000, 3)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 500, 16)        │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 500, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 250, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d_2              │ (None, 250, 32)        │           624 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 250, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 125, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv1d_3              │ (None, 125, 64)        │         2,208 │
│ (SeparableConv1D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 125, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_outputs (Dense)           │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,877 (22.96 KB)

 Trainable params: 5,653 (22.08 KB)

 Non-trainable params: 224 (896.00 B)


Training Lightweight 1D-CNN...
Epoch 1/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.4493 - macro_auc: 0.7763 - val_loss: 0.7773 - val_macro_auc: 0.7546 - learning_rate: 0.0010
Epoch 2/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.3634 - macro_auc: 0.8547 - val_loss: 0.6062 - val_macro_auc: 0.8354 - learning_rate: 0.0010
Epoch 3/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.3484 - macro_auc: 0.8666 - val_loss: 0.3609 - val_macro_auc: 0.8752 - learning_rate: 0.0010
Epoch 4/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.3398 - macro_auc: 0.8730 - val_loss: 0.3853 - val_macro_auc: 0.8737 - learning_rate: 0.0010
Epoch 5/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.3337 - macro_auc: 0.8784 - val_loss: 0.3484 - val_macro_auc: 0.8845 - learning_rate: 0.0010
Epoch 6/25
271/271 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.3295 - macro_auc: 0.8815 - val_loss: 0.3388 - val_macro_auc: 0.8903 - learning_rate: 0.0010
Epoch 7/25
271/271 ━━━━━━━━━━━━━━━━━━━

In [5]:
cnn_1d_probs = cnn_1d.predict(X_test, batch_size=128)
print(f"Lightweight 1D-CNN Macro AUC: {roc_auc_score(y_test, cnn_1d_probs, average='macro'):.4f}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Lightweight 1D-CNN Macro AUC: 0.8832


In [6]:
import numpy as np

# Save predictions
np.savez_compressed(RESULTS_DIR / "preds_1dcnn.npz", cnn1d=cnn_1d_probs)

# Save the full model architecture and weights
cnn_1d.save(MODELS_DIR / "tinyml_1d_cnn.keras")
print("Saved 1D-CNN predictions and model weights!")

Saved 1D-CNN predictions and model weights!
